# Quantization intro

In [1]:
import torch
import gc
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForCausalLM
from getpass import getpass
from huggingface_hub import login

/Users/macbookpro2023/Desktop/uni/year3/ai/small-LLM/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
login(token=getpass("Hugging Face token: "), add_to_git_credential=False)


In [3]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

In [4]:
MODEL_NAME = "google/gemma-3-270m-it"
DTYPE = torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE,).to(device)
model.eval()

print(f"Model loaded on: {model.device}\n")
print(model)

W1022 22:42:18.950000 74220 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 236/236 [00:01<00:00, 143.50it/s]


Model loaded on: mps:0

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RM

In [5]:
def tokenize_recieve(model, prompt):
    message = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(message, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    input_length = inputs["input_ids"].shape[1]
    new_tokens = outputs[0, input_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer

In [6]:
res = tokenize_recieve(model, "Explain machine learning in 2 simple sentences.")
print(res)

Machine learning is a type of computer science that uses algorithms to learn from data without being explicitly programmed.



In [7]:
def load_original_model():
    return AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(device).eval()

def clear_memory():
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

## Linear Quantization (symmetric and assymetric)

In [12]:
def symmetric_quantize(x: torch.Tensor, dtype=torch.int8) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Function implements Linear Symmetric Quantization per tensor.
    Zero point is fixed at 0.

    :param x: torch.Tensor, single tensor
    :param dtype: torch.dtype, type to downcast tensor to
    :return: tuple[torch.Tensor, torch.Tensor], downcast tensor, scale
    """
    max_original = x.abs().max()
    max_int = torch.iinfo(dtype).max
    s = max_original / max_int
    s = torch.where(s == 0, torch.ones_like(s), s)
    q = torch.round(x / s).clamp(-max_int, max_int).to(dtype)

    return q, s

def asymmetric_quantize(x: torch.Tensor, dtype=torch.int8) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Function implements Linear Asymmetric Quantization per tensor.

    :param x: torch.Tensor, single tensor
    :param dtype: torch.dtype, type to downcast tensor to
    :return: tuple[torch.Tensor, torch.Tensor, torch.Tensor], downcast tensor, scale, zero point
    """
    min_original, max_original = x.min().clamp(max=0), x.max().clamp(min=0)
    min_int, max_int = torch.iinfo(dtype).min, torch.iinfo(dtype).max

    s = (max_original - min_original) / (max_int - min_int)
    s = torch.where(s == 0, torch.ones_like(s), s)

    z = torch.round(min_int - min_original / s).clamp(min_int, max_int).to(dtype)
    q = (torch.round(x / s) + z.float()).clamp(min_int, max_int).to(dtype)

    return q, s, z

def dequantize(q: torch.Tensor, s: torch.Tensor, z: int | torch.Tensor = 0) -> torch.Tensor:
    """
    Function implements Linear Dequantization per tensor.

    :param q: torch.Tensor, quantized tensor
    :param s: torch.Tensor, scale
    :param z: torch.Tensor, zero point
    :return: torch.Tensor, dequantized tensor
    """
    return s.float() * (q.float() - torch.as_tensor(z, dtype=torch.float32, device=q.device))

def reconstruction_metrics(original: torch.Tensor, reconstructed: torch.Tensor) -> dict[str, float]:
    """
    Helper function to compute reconstruction losses between original and reconstructed tensors.
    :param original: torch.Tensor, original tensor
    :param reconstructed: torch.Tensor, reconstructed after quantization tensor
    :return: dict[str, float], metrics
    """
    mae = torch.abs(original - reconstructed).mean()
    mse = torch.square(original - reconstructed).mean()
    max_ae = torch.abs(original - reconstructed).max()

    return {"MAE": mae.item(), "MSE": mse.item(), "MaxAE": max_ae.item()}

In [13]:
@torch.no_grad()
def linear_weights_quantization(model, method="symmetric") -> list:
    """
    Function implements Linear Quantization on Gemma 3 model weights.

    :param model: Gemma 3 model
    :param method: str, symmetric or asymmetric
    :return: list, quantization results
    """
    results = []
    seen_params = set()

    embedding = model.get_input_embeddings()
    output_head = model.get_output_embeddings()

    for name, module in model.named_modules():

        if not (isinstance(module, torch.nn.Linear) or module is embedding or module is output_head):
            continue

        param = module.weight

        if id(param) in seen_params:
            continue

        seen_params.add(id(param))
        weights = param.detach().to(device="cpu", dtype=DTYPE).clone()

        if method == "symmetric":
            q, s = symmetric_quantize(weights)
            restored = dequantize(q, s)
            zero_point = 0
        else:
            q, s, z = asymmetric_quantize(weights)
            restored = dequantize(q, s, z)
            zero_point = z.item()

        metrics = reconstruction_metrics(weights, restored)

        results.append({
            "layer": name,
            "shape": tuple(weights.shape),
            "num_weights": weights.numel(),
            "scale": s.item(),
            "zero_point": zero_point,
            **{
                key: value.item() if torch.is_tensor(value) else value
                for key, value in metrics.items()
            },
        })

        param.copy_(restored.to(device=param.device, dtype=param.dtype))
        del weights, q, restored, s

    return results

In [14]:
prompts = [
    "Explain quantization of LLMs in two sentences.",
    "What is 12 multiplied by 4?"
]

del model
clear_memory()
model = load_original_model()
baseline_answers = [tokenize_recieve(model, p) for p in prompts]

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 330.90it/s]


In [15]:
results_sym = linear_weights_quantization(model, method="symmetric")
symmetric_answers = [tokenize_recieve(model, p) for p in prompts]

In [16]:
del model
clear_memory()
model = load_original_model()

results_asym = linear_weights_quantization(model, method="asymmetric")
asymmetric_answers = [tokenize_recieve(model, p) for p in prompts]

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 800.85it/s]


In [17]:
for prompt, baseline, symmetric, asymmetric in zip(prompts, baseline_answers, symmetric_answers, asymmetric_answers):
    print(f"Original prompt: {prompt}")
    print(f"Baseline output: {baseline}")
    print(f"Linear symmetric quantization output: {symmetric}")
    print(f"Linear asymmetric quantization output: {asymmetric}")

Original prompt: Explain quantization of LLMs in two sentences.
Baseline output: Quantization reduces the size of a language model, making it smaller and faster to run on hardware, such as GPUs or TPUs. This allows for more efficient inference and reduces computational costs.

Linear symmetric quantization output: Quantization reduces the size of a language model (LLM) by converting it to lower-precision formats, such as integers or floating-point numbers, to reduce memory usage and computational cost.

Linear asymmetric quantization output: Quantization refers to reducing the size and complexity of LLMs, often using techniques like pruning, distillation, and knowledge distillation, to reduce computational cost and improve inference speed.

Original prompt: What is 12 multiplied by 4?
Baseline output: 12 multiplied by 4 is 48.

Linear symmetric quantization output: 12 multiplied by 4 is 48.

Linear asymmetric quantization output: 12 * 4 = 48



## bitsandbytes